# Notebook 4 — Hyperparameter Tuning & Final Submission

**Input:** `features_train.csv`, `features_test.csv` (from Notebook 2)

**Output:** `final_submission.csv`

**Steps:**
1. Optuna hyperparameter search (XGBoost)
2. scale_pos_weight sweep
3. Feature selection (drop low-importance)
4. Final ensemble comparison
5. Generate submission

In [1]:
import os
import numpy as np
import pandas as pd
import optuna
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ============================================================
# Load engineered features
# ============================================================
train = pd.read_csv('features_train.csv')
test  = pd.read_csv('features_test.csv')

feature_cols = [c for c in train.columns
                if c not in ['uid', 'TARGET', 'NAME_CONTRACT_TYPE']]
for c in feature_cols:
    if c not in test.columns: test[c] = 0
test = test[['uid'] + feature_cols]

X = train[feature_cols]
y = train['TARGET']
spw = float((y == 0).sum() / (y == 1).sum())
cv = StratifiedKFold(5, shuffle=True, random_state=42)

SUBMISSION_DIR = r'E:\senior_ds_test\senior_ds_test\final_submission'

print(f'Features: {len(feature_cols)}  |  spw: {spw:.2f}')

e:\senior_ds_test\senior_ds_test\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Features: 69  |  spw: 11.41


## 1. Optuna Hyperparameter Search

In [2]:
def objective(trial):
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 300, 900),
        max_depth        = trial.suggest_int('max_depth', 3, 6),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        subsample        = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
        gamma            = trial.suggest_float('gamma', 0, 5),
        reg_alpha        = trial.suggest_float('reg_alpha', 0, 5),
        reg_lambda       = trial.suggest_float('reg_lambda', 0, 10),
        scale_pos_weight = spw,
        eval_metric      = 'auc',
        n_jobs           = -1,
        random_state     = 42,
    )
    model = XGBClassifier(**params)
    oof = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    return roc_auc_score(y, oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print(f'\nBEST CV AUC: {study.best_value:.5f}')
print('BEST PARAMS:', study.best_params)

[I 2026-09-15 02:12:27,427] A new study created in memory with name: no-name-a608f60c-08bc-4126-8529-9fbee37dd371
Best trial: 0. Best value: 0.677749:   2%|▎         | 1/40 [01:04<42:09, 64.85s/it]

[I 2026-09-15 02:13:32,280] Trial 0 finished with value: 0.6777492565127252 and parameters: {'n_estimators': 841, 'max_depth': 4, 'learning_rate': 0.022275998013210094, 'subsample': 0.9359215040384916, 'colsample_bytree': 0.8881985549921008, 'min_child_weight': 3, 'gamma': 3.100767253922067, 'reg_alpha': 0.11506610523160876, 'reg_lambda': 9.521814868811713}. Best is trial 0 with value: 0.6777492565127252.


Best trial: 0. Best value: 0.677749:   5%|▌         | 2/40 [01:47<32:57, 52.03s/it]

[I 2026-09-15 02:14:15,333] Trial 1 finished with value: 0.6770338868639967 and parameters: {'n_estimators': 477, 'max_depth': 4, 'learning_rate': 0.05686825408406451, 'subsample': 0.7417249849135318, 'colsample_bytree': 0.5468563742983557, 'min_child_weight': 1, 'gamma': 3.757158687929603, 'reg_alpha': 0.7251458159959007, 'reg_lambda': 4.292699183366528}. Best is trial 0 with value: 0.6777492565127252.


Best trial: 0. Best value: 0.677749:   8%|▊         | 3/40 [02:28<28:54, 46.88s/it]

...

[I 2026-09-15 02:45:13,978] Trial 37 finished with value: 0.6786615398500604 and parameters: {'n_estimators': 799, 'max_depth': 3, 'learning_rate': 0.0314973321775584, 'subsample': 0.7057052151445726, 'colsample_bytree': 0.9062437784959514, 'min_child_weight': 11, 'gamma': 3.8136424065582943, 'reg_alpha': 0.347664925223243, 'reg_lambda': 9.937663247930265}. Best is trial 32 with value: 0.6789282640095118.


Best trial: 32. Best value: 0.678928:  98%|█████████▊| 39/40 [33:51<01:00, 60.84s/it]

[I 2026-09-15 02:46:19,038] Trial 38 finished with value: 0.678837772562874 and parameters: {'n_estimators': 854, 'max_depth': 3, 'learning_rate': 0.03726959373208938, 'subsample': 0.6766126959506855, 'colsample_bytree': 0.9820016738356491, 'min_child_weight': 8, 'gamma': 4.144345926478017, 'reg_alpha': 1.1985086322751648, 'reg_lambda': 9.307937944746106}. Best is trial 32 with value: 0.6789282640095118.


Best trial: 39. Best value: 0.679069: 100%|██████████| 40/40 [34:52<00:00, 52.32s/it]

[I 2026-09-15 02:47:20,091] Trial 39 finished with value: 0.6790688578702062 and parameters: {'n_estimators': 846, 'max_depth': 3, 'learning_rate': 0.0293794138331454, 'subsample': 0.6749056484741371, 'colsample_bytree': 0.8976598451481258, 'min_child_weight': 9, 'gamma': 4.165802788078908, 'reg_alpha': 0.7198236909715666, 'reg_lambda': 8.18919307965033}. Best is trial 39 with value: 0.6790688578702062.

BEST CV AUC: 0.67907
BEST PARAMS: {'n_estimators': 846, 'max_depth': 3, 'learning_rate': 0.0293794138331454, 'subsample': 0.6749056484741371, 'colsample_bytree': 0.8976598451481258, 'min_child_weight': 9, 'gamma': 4.165802788078908, 'reg_alpha': 0.7198236909715666, 'reg_lambda': 8.18919307965033}


In [3]:
# Store best params for subsequent steps
BEST = {**study.best_params,
        'scale_pos_weight': spw,
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1}
print('BEST config:', BEST)

BEST config: {'n_estimators': 846, 'max_depth': 3, 'learning_rate': 0.0293794138331454, 'subsample': 0.6749056484741371, 'colsample_bytree': 0.8976598451481258, 'min_child_weight': 9, 'gamma': 4.165802788078908, 'reg_alpha': 0.7198236909715666, 'reg_lambda': 8.18919307965033, 'scale_pos_weight': 11.41311677826851, 'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1}


## 2. scale_pos_weight Sweep

In [4]:
def cv_auc(params, cols):
    m = XGBClassifier(**params)
    oof = cross_val_predict(m, train[cols], y, cv=cv,
                            method='predict_proba', n_jobs=-1)[:,1]
    return roc_auc_score(y, oof), oof

print('=== scale_pos_weight sweep ===')
best_spw, best_auc = spw, 0
for w in [1.0, spw**0.5, spw*0.5, spw]:
    a, _ = cv_auc({**BEST, 'scale_pos_weight': w}, feature_cols)
    print(f'  spw={w:6.2f}  AUC={a:.5f}')
    if a > best_auc:
        best_auc, best_spw = a, w

BEST['scale_pos_weight'] = best_spw
print(f'\n→ Best spw={best_spw:.2f}  AUC={best_auc:.5f}')

=== scale_pos_weight sweep ===
  spw=  1.00  AUC=0.67991
  spw=  3.38  AUC=0.67962
  spw=  5.71  AUC=0.67951
  spw= 11.41  AUC=0.67907

→ Best spw=1.00  AUC=0.67991


## 3. Feature Selection (Drop Low-Importance)

In [5]:
print('=== Feature Selection ===')
m = XGBClassifier(**BEST).fit(train[feature_cols], y)
imp = pd.Series(m.feature_importances_, index=feature_cols)

base_auc, _ = cv_auc(BEST, feature_cols)
print(f'  All {len(feature_cols)} feats: AUC={base_auc:.5f}')

best_cols, best_sel = feature_cols, base_auc
for q in [0.10, 0.20, 0.30]:
    keep = imp[imp > imp.quantile(q)].index.tolist()
    a, _ = cv_auc(BEST, keep)
    print(f'  Drop bottom {int(q*100)}% → {len(keep)} feats: AUC={a:.5f}')
    if a > best_sel:
        best_sel, best_cols = a, keep

print(f'\n→ Kept {len(best_cols)} feats  AUC={best_sel:.5f}')
print('Selected features:', best_cols)

=== Feature Selection ===
  All 69 feats: AUC=0.67991
  Drop bottom 10% → 62 feats: AUC=0.68008
  Drop bottom 20% → 55 feats: AUC=0.67986
  Drop bottom 30% → 48 feats: AUC=0.68025

→ Kept 48 feats  AUC=0.68025
Selected features: ['is_cash_loan', 'acc_n_accounts', 'acc_n_open', 'acc_loan_amt_sum', 'acc_loan_amt_mean', 'acc_loan_amt_max', 'acc_loan_amt_std', 'acc_log_loan_sum', 'acc_log_loan_mean', 'acc_overdue_sum', 'acc_overdue_max', 'acc_days_since_open_min', 'acc_days_since_open_max', 'acc_days_since_open_mean', 'acc_days_since_close_min', 'acc_open_ratio', 'acc_overdue_ratio', 'acc_overdue_amt_ratio', 'acc_zero_amt_ratio', 'acc_cnt_Car loan', 'acc_cnt_Consumer credit', 'acc_cnt_Credit card', 'acc_cnt_Microloan', 'acc_cnt_Mortgage', 'acc_cnt_Car loan_share', 'acc_cnt_Consumer credit_share', 'acc_cnt_Credit card_share', 'acc_cnt_Microloan_share', 'acc_cnt_Mortgage_share', 'acc_cnt_Other_share', 'pmt_recent_dpd_max', 'pmt_recent3_max', 'pmt_total_months', 'pmt_late_ratio_mean', 'pmt_n_

## 4. Final Ensemble Comparison

In [6]:
print('=== Ensemble on selected features ===')
Xtr = train[best_cols]

# XGBoost (tuned)
xgb_final = XGBClassifier(**BEST)
oof_x = cross_val_predict(xgb_final, Xtr, y, cv=cv,
                           method='predict_proba', n_jobs=-1)[:,1]
auc_x = roc_auc_score(y, oof_x)

# LightGBM
lgb_final = LGBMClassifier(
    n_estimators=500, max_depth=3, num_leaves=8,
    learning_rate=0.05, subsample=0.75, colsample_bytree=0.65,
    reg_alpha=3.5, reg_lambda=8.5, min_child_samples=40,
    scale_pos_weight=best_spw, random_state=42, n_jobs=-1, verbose=-1
)
oof_l = cross_val_predict(lgb_final, Xtr, y, cv=cv,
                           method='predict_proba', n_jobs=-1)[:,1]
auc_l = roc_auc_score(y, oof_l)

# CatBoost
cat_final = CatBoostClassifier(
    iterations=500, depth=3, learning_rate=0.05,
    l2_leaf_reg=8, subsample=0.75,
    scale_pos_weight=best_spw, random_state=42,
    verbose=0, allow_writing_files=False
)
oof_c = cross_val_predict(cat_final, Xtr, y, cv=cv,
                           method='predict_proba', n_jobs=1)[:,1]
auc_c = roc_auc_score(y, oof_c)

print(f'  XGB={auc_x:.5f}  LGB={auc_l:.5f}  CAT={auc_c:.5f}')

# Rank blend
rank_blend = (rankdata(oof_x) + rankdata(oof_l) + rankdata(oof_c)) / 3
auc_blend = roc_auc_score(y, rank_blend)
print(f'  RANK-BLEND AUC={auc_blend:.5f}')

# Pick the best strategy
use_blend = auc_blend > auc_x
final_auc = auc_blend if use_blend else auc_x
strategy = 'rank-blend' if use_blend else 'xgb-only'
print(f'\n🏆 FINAL CV AUC = {final_auc:.5f}  ({strategy})')

=== Ensemble on selected features ===
  XGB=0.68025  LGB=0.67937  CAT=0.67762
  RANK-BLEND AUC=0.67983

🏆 FINAL CV AUC = 0.68025  (xgb-only)


## 5. Generate Submission

In [7]:
print('=== Fitting on full training data ===')
Xtest = test[best_cols]

# Always fit XGBoost (used in both strategies)
xgb_full = XGBClassifier(**BEST).fit(Xtr, y)
pred_x = xgb_full.predict_proba(Xtest)[:,1]

if use_blend:
    print('Using rank-blend of 3 models...')
    lgb_full = lgb_final.fit(Xtr, y)
    pred_l = lgb_full.predict_proba(Xtest)[:,1]

    cat_full = cat_final.fit(Xtr, y)
    pred_c = cat_full.predict_proba(Xtest)[:,1]

    # Rank-blend normalised to [0, 1]
    test_pred = (rankdata(pred_x) + rankdata(pred_l) + rankdata(pred_c)) / (3 * len(pred_x))
else:
    print('Using XGBoost only...')
    test_pred = pred_x

print(f'Test predictions — min: {test_pred.min():.4f}  max: {test_pred.max():.4f}  mean: {test_pred.mean():.4f}')

=== Fitting on full training data ===
Using XGBoost only...
Test predictions — min: 0.0038  max: 0.8044  mean: 0.0802


In [8]:
# Create submission DataFrame
sub = pd.DataFrame({'uid': test['uid'], 'TARGET': test_pred})

# Save
os.makedirs(SUBMISSION_DIR, exist_ok=True)
out_path = os.path.join(SUBMISSION_DIR, 'final_submission.csv')
sub.to_csv(out_path, index=False)

print(f'\n✅ Submission saved: {out_path}')
print(f'   Shape: {sub.shape}')
print(f'   CV AUC: {final_auc:.5f}')
print(sub.head(10))


✅ Submission saved: E:\senior_ds_test\senior_ds_test\final_submission\final_submission.csv
   Shape: (46127, 2)
   CV AUC: 0.68025
           uid    TARGET
0  CMO22835242  0.044608
1  MRJ34316727  0.165255
2  UAV00534378  0.079647
3  IPQ08190402  0.058909
4  NQN84331006  0.061076
5  LPS50307411  0.038565
6  TSM49103703  0.116379
7  FEX47174606  0.057191
8  MVK91941657  0.060144
9  VTQ11003334  0.043260


In [9]:
# Final validation checks
print('=== Final Checks ===')
print(f'Unique UIDs in submission: {sub["uid"].nunique()}')
print(f'Any NaN in TARGET: {sub["TARGET"].isna().any()}')
print(f'TARGET range: [{sub["TARGET"].min():.6f}, {sub["TARGET"].max():.6f}]')
print(f'\nTARGET distribution:')
print(sub['TARGET'].describe())

=== Final Checks ===
Unique UIDs in submission: 46127
Any NaN in TARGET: False
TARGET range: [0.003803, 0.804373]

TARGET distribution:
count    46127.000000
mean         0.080250
std          0.054035
min          0.003803
25%          0.045007
50%          0.067029
75%          0.099595
max          0.804373
Name: TARGET, dtype: float64


## Summary

| Step | Detail |
|------|--------|
| Optuna tuning | 40 trials on XGBoost |
| SPW sweep | Tested 1.0, √spw, spw/2, spw |
| Feature selection | Dropped bottom 10/20/30% by importance |
| Ensemble | XGB vs LGB vs CAT vs Rank-Blend |
| Final strategy | Whichever scored highest on 5-fold CV AUC |
| Submission | `final_submission.csv` |